In [ ]:
from datetime import datetime
from opera_utils import disp
import geopandas as gpd
import os
from pathlib import Path
import subprocess
from transboundary_opera import displacement_tools as dt
import matplotlib.pyplot as plt
import xarray as xr
from shapely.geometry import mapping
import ultraplot as uplt
from pyproj import CRS
import asf_search as asf
import re

# %matplotlib widget
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
SEARCH_START = datetime(2024, 6, 1)
SEARCH_END = datetime(2025, 1, 1)

gdf = gpd.read_file(Path(os.getcwd()).parent.parent / "raw_data/TBA_full.shp")
# frame_ids = dt.get_opera_frame_ids(Path(os.getcwd()).parent.parent / "raw_data/TBA_full.shp")     # this only gets one set of frame ids 

In [ ]:
# Initialize a list to store frame IDs for each row
all_frame_ids = []

for entry, row in gdf.iterrows():
    
    results = asf.geo_search(
        intersectsWith=row.geometry.convex_hull.wkt,
        dataset=asf.DATASET.OPERA_S1,
        processingLevel=asf.PRODUCT_TYPE.DISP_S1,
        maxResults=50  # We only need a few results to identify the Frame IDs
    )
    
    # Set for the current row
    current_frame_ids = set()
    pattern = re.compile(r'_F(\d{5})_')

    for product in results:
        filename = product.properties['fileName']
        match = pattern.search(filename)
        if match:
            frame_id_str = match.group(1) 
            current_frame_ids.add(int(frame_id_str))
    
    # Add the sorted list of frame IDs to our collection
    all_frame_ids.append(sorted(list(current_frame_ids)))

# Assign the collected frame IDs to a new column in the GeoDataFrame
gdf['frame_ids'] = all_frame_ids

In [ ]:
# TODO: attempt refined download for each of the frames that cover each aquifer. May need to clip the bbox of the aquifers and segment by the frame_ids

unique_frame_ids = sorted({fid for ids in gdf['frame_ids'] if isinstance(ids, (list, tuple, set)) for fid in ids})
len(unique_frame_ids)

In [ ]:
disp_products = {}

for frame_id in unique_frame_ids:
    disp_products[frame_id] = disp.search(
        frame_id=frame_id,
        start_datetime=SEARCH_START,
        end_datetime=SEARCH_END,
        url_type="https",
        print_urls=False,
    )

# test_frame_id = unique_frame_ids[21]

# # individual search for one frame id
# disp_products = disp.search(
#         frame_id=test_frame_id,
#         start_datetime=datetime(2024, 1, 1),
#         end_datetime=datetime(2025, 1, 1),
#         url_type="https",
#         print_urls=False,
#     )

# disp_products_stack = disp.DispProductStack(disp_products)

In [ ]:
tot_bytes = 0

for key, list in disp_products.items():
    # print(f"Frame ID: {key}, Number of Products: {len(list)}")
    for disp_product in list:
        tot_bytes += disp_product.size_in_bytes

print(f"\nTotal Storage needed for OPERA DISP Products: {round(tot_bytes / 1073741824, 2)} GB")

In [ ]:
# if you have a .netrc or appropriate envronment variables set up, you can directly open a file or two in memory
# test1 = xr.open_dataset(disp.open_file(url = disp_products[0].filename))
# test2 = xr.open_dataset(disp.open_file(url = disp_products[1].filename))

# Download Data

In [ ]:
# out_path = Path('/Users/clayc/Documents/Work/US_Mex_InSAR') / "OPERA_data"
out_path = Path('/home/clayc/Documents/US_Mex_InSAR') / "OPERA_data"
os.makedirs(out_path, exist_ok=True)

for key, list in disp_products.items():
    print(f"Frame ID: {key}, Number of Products: {len(list)}")

    download_dir = out_path / str(key)
    os.makedirs(download_dir, exist_ok=True)

    if len(list) == 0:
        print(f"No products found for Frame ID: {key}. Skipping download.")
        continue


    cmd = [
        "opera-utils", "disp-s1-download",
        "--output-dir", str(download_dir),
        "--frame-id", str(key),
        "--start-datetime", SEARCH_START.strftime('%Y-%m-%d'),
        "--end-datetime", SEARCH_END.strftime('%Y-%m-%d'),
        # "--bbox", f"{round(bbox[0],2)}", f"{round(bbox[1],2)}", f"{round(bbox[2],2)}", f"{round(bbox[3],2)}",
        "--url-type", "HTTPS", 
        "--num-workers", "8"
    ]

    print("Starting download...")
    subprocess.run(cmd, check=True)

# Testing the opera-utils create-mintpy.sh function! I think this will download, stack, and move the OPERA disp products to h5 compatible files for MintPy LOS decomposition
- probably needs to be a subprocess run